# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates the loading, exploration, and processing of the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. This includes metadata review, tabular data loading via Croissant schema, data processing steps, and example visualizations—all referencing entities by their Croissant `@id` fields.

### Dataset Source

The dataset is described by its Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install the mlcroissant library, if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Dataset using the Croissant schema
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")

## 2. Data Overview

List available record sets and their structure by `@id`. This review helps us know what tabular data and fields can be loaded from the package.

We'll print available record sets, fields, and columns—all referenced by their `@id` as required by the Croissant standard.

In [ ]:
# List the available record sets in the Croissant package

print("Available Record Sets (by @id):")
for record_set in dataset.record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name', '(no name)')}")

# For each record set, also list its fields by their @id
print("\nRecord Set Fields (by @id):")
for record_set in dataset.record_sets:
    print(f"\nRecord Set: {record_set['@id']}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, str):
            print(f"  - {field}")
        elif isinstance(field, dict):
            print(f"  - {field.get('@id', '(no @id)')}: {field.get('name', '(no name)')}")

# Print the @id of the first record set to use for the concrete example below
if dataset.record_sets:
    first_record_set_id = dataset.record_sets[0]['@id']
    print(f"\n[INFO] Using the first record set for examples: {first_record_set_id}")
else:
    first_record_set_id = None
    print("[ERROR] No record sets found in the Croissant metadata!")

## 3. Data Extraction

Load the data for each record set into a Pandas DataFrame. All references will use the Croissant `@id`. We'll demonstrate with the first record set identified above.

In [ ]:
# Extract data from each record set via its @id

record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set '@id': {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"[Warning] Could not load records for {record_set_id}: {e}")

# Show the columns of the first record set DataFrame
if first_record_set_id and first_record_set_id in dataframes:
    first_df = dataframes[first_record_set_id]
    print("\nFirst DataFrame columns:")
    print(first_df.columns.tolist())
    first_df.head()
else:
    print("[ERROR] No DataFrame available for the first record set.")

## 4. Exploratory Data Analysis (EDA)

Select a numeric field (by `@id`) from the loaded DataFrame for analysis: filter, normalize, and group. All references continue by Croissant `@id`.

*If you want to use a different field or grouping, check the DataFrame's columns above and use the appropriate `@id`.*

In [ ]:
# Pick a numeric field by inspecting the columns of the first DataFrame
# For demonstration, we'll select the first column with numeric-looking data (e.g., age, interval length)

df = dataframes.get(first_record_set_id)

if df is not None and not df.empty:
    # Attempt to auto-detect a numeric column by dtype or content
    numeric_field_ids = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if not numeric_field_ids:
        # Try to parse columns that look like numbers (e.g., strings of digits)
        for col in df.columns:
            try:
                # If many values are digits, treat as numeric
                if df[col].astype(str).str.isdigit().sum() > len(df) // 2:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    numeric_field_ids.append(col)
            except Exception:
                continue

    if numeric_field_ids:
        numeric_field_id = numeric_field_ids[0]
        print(f"[INFO] Selected numeric field '@id': {numeric_field_id}")
        # Set a demonstration threshold, e.g. age > 50 or interval > 10
        # For a real EDA, adjust threshold as appropriate for the selected field
        threshold = df[numeric_field_id].dropna().median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records ({numeric_field_id} > {threshold}): {filtered_df.shape[0]} rows")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize the numeric field for filtered results
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt to select a grouping field (e.g. sex, diagnosis type) by @id
        # We'll pick the first non-numeric, non-unique field
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object and df[col].nunique() < len(df) // 2:
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[[numeric_field_id, norm_col]].mean()
            print(f"\nGrouped (mean) by '{group_field}':")
            print(grouped_df.head())
        else:
            print("[INFO] No suitable categorical field found for grouping.")
    else:
        print("[WARNING] No numeric-like fields found in the record set.")
else:
    print("[ERROR] No data found in selected record set, cannot proceed with EDA.")

## 5. Visualization

Visualize the distribution of the filtered numeric field, and, if available, stratify by the group field. Make sure plots are labeled with the correct Croissant `@id` references in titles and axes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and 'numeric_field_id' in locals():
    fig, ax = plt.subplots(figsize=(8,4))
    # Distribution in original data
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True, ax=ax)
    ax.set_title(f"Distribution of field '@id': {numeric_field_id}")
    ax.set_xlabel(numeric_field_id)
    plt.show()

    # If a group field is available, show boxplots by group
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"'{numeric_field_id}' grouped by '{group_field}' (@id)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion

In this notebook, we successfully loaded and explored the FAIR^2 clinical oncology dataset using the Croissant schema and the `mlcroissant` library. All tabular entities, record sets, and fields were referenced by their Croissant `@id`, ensuring reproducibility and standardization. Example analyses included filtering on numeric variables, normalization, and group aggregation. 

*For more in-depth domain interpretation (e.g., the meaning of field `@id`s), or for advanced modeling, consult the full Croissant metadata and associated documentation.*